# tarea_3_teoria

- Villasante López, Bryan Jean Pierre

## Tarea: Análisis de Datos Distribuidos con PySpark

**Instrucciones:** Utiliza el notebook de Databricks y el lenguaje PySpark para responder a las siguientes consultas sobre la tabla *samples.bakehouse.sales_customers*. **No se permite el uso de Python nativo (bucles for/if) ni conversiones a Pandas.**

## Nivel I: Exploración de Datos (Básico)

### 1. Carga de Datos: Escribe el código para cargar la tabla sales_customers en un DataFrame llamado df y muestra las primeras 5 filas.

In [0]:
df = spark.read.table("samples.bakehouse.sales_customers")
df.write.mode("overwrite").saveAsTable("bronce_dev.bakehouse.customer_brz")

In [0]:
display(spark.table("bronce_dev.bakehouse.customer_brz").limit(5))


### 2. Conteo de Registros: ¿Cuántos clientes totales existen en esta base de datos?

In [0]:
total_clientes = spark.table("bronce_dev.bakehouse.customer_brz").count()
display(total_clientes)

### 3. Inspección de Esquema: Muestra el esquema (schema) de la tabla para identificar qué columnas son de tipo cadena (string) y cuáles son numéricas (long).

In [0]:
schema = spark.table("bronce_dev.bakehouse.customer_brz").schema
string_cols = [f.name for f in schema.fields if f.dataType.typeName() == "string"]
long_cols = [f.name for f in schema.fields if f.dataType.typeName() == "long"]

print("Columnas de tipo cadena (string):\n   ", string_cols)
print("\nColumnas de tipo numérico (long):\n   ", long_cols)

### 4. Filtro Geográfico: Filtra el DataFrame para mostrar solo a los clientes que viven en el país 'Japan'.

In [0]:
df_japan = spark.table("bronce_dev.bakehouse.customer_brz").filter("country = 'Japan'")
display(df_japan)

### 5. Selección Específica: Genera una tabla que solo muestre las columnas first_name, last_name y email_address.

In [0]:
df_selected = spark.table("bronce_dev.bakehouse.customer_brz").select("first_name", "last_name", "email_address")
display(df_selected)

### 6. Búsqueda de Nulos: Escribe una consulta para verificar si existen valores nulos en la columna city.

In [0]:
df_null_city = spark.table("bronce_dev.bakehouse.customer_brz").filter("city IS NULL")
display(df_null_city)

## Nivel II: Agregaciones y Ordenamiento (Intermedio)

### 7. Ranking de Países: ¿Cuáles son los 5 países con mayor cantidad de clientes? Muestra el país y el conteo ordenado de mayor a menor.

In [0]:
df_country_count = (
    spark.table("bronce_dev.bakehouse.customer_brz")
    .groupBy("country")
    .count()
    .orderBy("count", ascending=False)
    .limit(5)
)
display(df_country_count)

### 8. Análisis de Género: Agrupa a los clientes por la columna gender y muestra cuántos hay de cada uno.

In [0]:
df_gender_count = (
    spark.table("bronce_dev.bakehouse.customer_brz")
    .groupBy("gender")
    .count()
    .orderBy("count", ascending=False)
)
display(df_gender_count)

### 9. Limpieza de Texto: Muestra los nombres de los clientes (first_name) convirtiendo todos los textos a Mayúsculas.

In [0]:
from pyspark.sql.functions import upper

df_upper_names = spark.table("bronce_dev.bakehouse.customer_brz").select(upper("first_name").alias("FIRST_NAME"))
display(df_upper_names)

### 10. Orden Alfabético: Obtén una lista de los primeros 10 clientes ordenados alfabéticamente por su apellido (last_name).

In [0]:
df_sorted_lastname = (
    spark.table("bronce_dev.bakehouse.customer_brz")
    .orderBy("last_name")
    .limit(10)
)
display(df_sorted_lastname)

### 11. Dominios de Correo: Filtra a todos los clientes cuyo correo electrónico (email_address) contenga la palabra 'example'.

In [0]:
df_example_email = spark.table("bronce_dev.bakehouse.customer_brz").filter("email_address LIKE '%example%'")
display(df_example_email)

### 12. Conteo de Continentes: ¿Cuántos continentes diferentes aparecen en la base de datos? Únicamente muestra el valor numérico.

In [0]:
num_continentes = spark.table("bronce_dev.bakehouse.customer_brz").select("continent").distinct().count()
display(num_continentes)

## Nivel III: Transformaciones y Lógica de Negocio (Avanzado)

### 13. Columna Derivada: Crea una nueva columna llamada nombre_completo que concatene el nombre y el apellido con un espacio de por medio.

In [0]:
from pyspark.sql.functions import concat_ws

df_nombre_completo = spark.table("bronce_dev.bakehouse.customer_brz").withColumn(
    "nombre_completo", concat_ws(" ", "first_name", "last_name")
)
display(df_nombre_completo)

### 14. Renombramiento: Cambia el nombre de la columna postal_zip_code por codigo_postal y muestra el resultado.

In [0]:
df_renamed = spark.table("bronce_dev.bakehouse.customer_brz").withColumnRenamed("postal_zip_code", "codigo_postal")
display(df_renamed)

### 15. Clasificación por Región: Crea una columna llamada es_australia que contenga el valor booleano True si el país es 'Australia' y False en caso contrario.

In [0]:
from pyspark.sql.functions import when, col

df_es_australia = spark.table("bronce_dev.bakehouse.customer_brz").withColumn(
    "es_australia", when(col("country") == "Australia", True).otherwise(False)
)
display(df_es_australia)

### 16. Eliminación de Datos: Genera un nuevo DataFrame que contenga toda la información original excepto la columna phone_number.

In [0]:
df_sin_phone = spark.table("bronce_dev.bakehouse.customer_brz").drop("phone_number")
display(df_sin_phone)

### 17. Cálculo de Proporción: ¿Qué porcentaje del total de clientes pertenece al continente 'Asia'?

In [0]:
df_asia = spark.table("bronce_dev.bakehouse.customer_brz")
total_clientes = df_asia.count()
clientes_asia = df_asia.filter(col("continent") == "Asia").count()
porcentaje_asia = (clientes_asia / total_clientes) * 100
display(porcentaje_asia)

### 18. Estadísticas de ID: Utiliza la función describe() para obtener un resumen estadístico de la columna customerID.

In [0]:
df_describe = spark.table("bronce_dev.bakehouse.customer_brz").select("customerID").describe()
display(df_describe)

## Nivel IV: Integración y Eficiencia (Reto Final)

### 19. Join de Tablas: (Investigación) Escribe el código necesario para unir (join) la tabla sales_customers con sales_transactions usando la columna customerID.

In [0]:
df_customers = spark.read.table("samples.bakehouse.sales_customers")
df_transactions = spark.read.table("samples.bakehouse.sales_transactions")

df_joined = df_customers.join(df_transactions, on="customerID", how="inner")
display(df_joined)

### 20. Persistencia: ¿Cómo guardarías el resultado de los clientes filtrados de 'Japan' como una nueva tabla llamada clientes_japon_tu_apellido en el catálogo de Databricks?

In [0]:
df_japan = spark.table("bronce_dev.bakehouse.customer_brz").filter(col("country") == "Japan")
df_japan.write.mode("overwrite").saveAsTable("clientes_japon_Villasante")